# Pydantic Serialization: Exporting Validated Data

Once validation succeeds, exporting schemas to standard python dictionaries or serializing to JSON format is standard practice. Pydantic provides optimized, built-in methods to serialize data with filters.

In this notebook, we cover:
1. `model_dump()`: Converting models to standard Python dicts.
2. Filtering exports using `include` and `exclude`.
3. Omitting default or empty parameters using `exclude_unset`.
4. `model_dump_json()`: Serializing models to JSON strings.


In [1]:
!uv pip install pydantic 'pydantic[email]' --quiet


## 1. Imports and Class Definitions


In [2]:
from pydantic import BaseModel, EmailStr, AnyUrl, Field, field_validator, model_validator, computed_field
from typing import List, Dict, Optional, Annotated


In [3]:
class ContactDetails(BaseModel):
    email_id: Annotated[EmailStr, Field(description='Primary email address of the patient', examples=['john@example.com'])]
    contact_number: Annotated[str, Field(min_length=10, max_length=15, description='Primary contact number', examples=['9999999999'])]
    emergency_contact_number: Annotated[Optional[str], Field(default=None, min_length=10, max_length=15, description='Emergency contact number', examples=['8888888888'])]

    @field_validator('email_id')
    @classmethod
    def email_validator(cls, value: str) -> str:
        valid_domain = ['domain.io', 'example.com']
        domain_name = value.split("@")[-1]

        if domain_name not in valid_domain:
            raise ValueError('Not a valid domain')
        
        return value


In [4]:
class AddressDetails(BaseModel):
    city: str
    state: str
    postalCode: str


In [5]:
class Patient(BaseModel):

    name: str = Annotated[str, Field(max_length=150, title='Name of the patient', description='Patient Name for records', examples=['John Doe'])]
    age: int
    linkedin_url: Optional[AnyUrl] = None
    weight: Annotated[float, Field(gt=0, description='Submit patient weight for the report', strict=True)]
    height: Annotated[float, Field(gt=0, description='Submit patient height for the report', strict=True)]
    married: Annotated[bool, Field(default=None, description='Is the patient married or not')]
    allergies: Annotated[Optional[List[str]], Field(default=None, max_length=5)]
    contact_details: ContactDetails
    address_details: AddressDetails

    @field_validator('name')
    @classmethod
    def transform_name(cls, value: str) -> str:
        return value.upper()
    
    @field_validator('age', mode='before')
    @classmethod
    def validate_age(cls, value: int) -> int:
        if 0 < value < 120:
            return value
        else:
            raise ValueError("Age should be in between 0 and 120")

    @model_validator(mode='after')
    def validate_emergency_contact(self):
        if self.age > 60 and self.contact_details.emergency_contact_number is None:
            raise ValueError('Emergency contact number is mandatory for patients above 60 years of age')
        return self
    
    @computed_field
    @property
    def calculate_bmi(self) -> float:
        bmi = round(self.weight / (self.height**2), 2)
        return bmi


## 2. Helper Pipelines and Mock Inputs


In [6]:
def insert_patient_data(patient: Patient):
    print(f"Patient Name: {patient.name}")
    print(f"Patient Age: {patient.age}")
    print(f"Patient Weight: {patient.weight}")
    print(f"Patient Married Status: {patient.married}")
    print(f"Patient Allergies: {patient.allergies}")
    print(f"Patient Contact Details: {patient.contact_details}")
    print(f"Patient Address Details: {patient.address_details}")
    print(f"Patient Calculated BMI: {patient.calculate_bmi}")
    print('Patient info inserted')


In [7]:
def update_patient_data(patient: Patient):
    print(f"Patient Name: {patient.name}")
    print(f"Patient Age: {patient.age}")
    print(f"Patient Weight: {patient.weight}")
    print(f"Patient Married Status: {patient.married}")
    print(f"Patient Allergies: {patient.allergies}")
    print(f"Patient Contact Details: {patient.contact_details}")
    print(f"Patient Calculated BMI: {patient.calculate_bmi}")
    print(f"Patient Address Details: {patient.address_details}")
    print('Patient info updateed')


In [8]:
address_detail = {
    'city': 'Gurugram',
    'state': 'Haryana',
    'postalCode': '122021'
}


In [9]:
contact_details = {
    'email_id': 'example@domain.io',
    'contact_number': '9999999999'
}


In [10]:
patient_info = {
    'name': 'Kevin',
    'age': 26,
    'weight': 67.9,
    'height': 1.73,
    'married': False,
    'allergies': ['lactose', 'dust'],
    'contact_details': contact_details,
    'address_details': address_detail
}


## 3. Instantiating the Model


In [11]:
patient = Patient(**patient_info)
patient


Patient(name='KEVIN', age=26, linkedin_url=None, weight=67.9, height=1.73, married=False, allergies=['lactose', 'dust'], contact_details=ContactDetails(email_id='example@domain.io', contact_number='9999999999', emergency_contact_number=None), address_details=AddressDetails(city='Gurugram', state='Haryana', postalCode='122021'), calculate_bmi=22.69)

## 4. Exporting to a Dictionary with Filters

Pydantic's `model_dump()` converts the model instance recursively to standard python types:
*   `exclude`: Removes specified keys from the output dictionary (e.g. `address_details`).
*   `exclude_unset`: If `True`, fields that were not explicitly provided in the input payload (and just fell back to defaults, like `linkedin_url` or optional defaults) are omitted from the export dictionary.
*   `include`: Selects only a specific subset of fields to serialize.


In [17]:
# Exclude address_details, exclude fields that were not explicitly set (such as linkedin_url)
patient_dict = patient.model_dump(exclude={'address_details'}, exclude_unset=True)
print("Type:", type(patient_dict))
patient_dict


Type: <class 'dict'>


{'name': 'KEVIN',
 'age': 26,
 'weight': 67.9,
 'height': 1.73,
 'married': False,
 'allergies': ['lactose', 'dust'],
 'contact_details': {'email_id': 'example@domain.io',
  'contact_number': '9999999999'},
 'calculate_bmi': 22.69}

## 5. Serializing directly to a JSON String

`model_dump_json()` directly exports your model as a highly optimized JSON string.


In [18]:
patient_json = patient.model_dump_json()
print("Type:", type(patient_json))
patient_json


Type: <class 'str'>


'{"name":"KEVIN","age":26,"linkedin_url":null,"weight":67.9,"height":1.73,"married":false,"allergies":["lactose","dust"],"contact_details":{"email_id":"example@domain.io","contact_number":"9999999999","emergency_contact_number":null},"address_details":{"city":"Gurugram","state":"Haryana","postalCode":"122021"},"calculate_bmi":22.69}'

In [14]:
insert_patient_data(patient=patient)


Patient Name: KEVIN
Patient Age: 26
Patient Weight: 67.9
Patient Married Status: False
Patient Allergies: ['lactose', 'dust']
Patient Contact Details: email_id='example@domain.io' contact_number='9999999999' emergency_contact_number=None
Patient Address Details: city='Gurugram' state='Haryana' postalCode='122021'
Patient Calculated BMI: 22.69
Patient info inserted


In [15]:
update_patient_data(patient=patient)


Patient Name: KEVIN
Patient Age: 26
Patient Weight: 67.9
Patient Married Status: False
Patient Allergies: ['lactose', 'dust']
Patient Contact Details: email_id='example@domain.io' contact_number='9999999999' emergency_contact_number=None
Patient Calculated BMI: 22.69
Patient Address Details: city='Gurugram' state='Haryana' postalCode='122021'
Patient info updateed
